# Etapa F — API de inferencia SageMaker para BTS (LightGBM float64)

**Finalidad:** publicar el **modelo definitivo tras Optuna y refit** como un endpoint de inferencia en tiempo real. **No entrena** ni recalcula Gold.

Este notebook ejecuta un script auxiliar independiente `bts_sagemaker_endpoint.py`. Cada operación se puede repetir después de reiniciar AWS Academy Learner Lab sin duplicar modelos ni endpoints.

**Prerequisitos:**
- Ejecutar en `SageMaker/src/` con la sesión del Learner Lab activa (credenciales renovadas).
- Tener subido a S3 `models/flight-delay/<model_version>/model.tar.gz` y `modeling/tuning/lightgbm-float64-seed42-h240/model_version.json`, desde la sección 11 del tuning.
- Subir `bts_sagemaker_endpoint.py` **a la misma carpeta que este notebook**. Se necesita `boto3` (ya disponible normalmente).
- `LabRole` debe autorizar `sagemaker:CreateModel`, `sagemaker:CreateEndpointConfig`, `sagemaker:CreateEndpoint`, `sagemaker:InvokeEndpoint`, `iam:PassRole` y acceso al S3 correspondiente. Los permisos efectivos de AWS Academy pueden limitar recursos.

**Modelo:** predice `DepDel15` (salida retrasada ≥ 15 min) **240 min antes de la salida**; las características históricas/operativas deben estar ya calculadas. Un JSON con solo aerolínea, origen y hora no es suficiente.

> **Costos:** un endpoint **real-time** factura mientras existe, aunque apagues SageMaker Notebook. Tras mostrarlo o probarlo, ejecuta el paso de limpieza cuando ya no lo necesites.


## 0. Comprobar el entorno

Verifica que la ruta sea `src`, que el script auxiliar esté subido y que Boto3 esté disponible. Esta celda no modifica AWS.

In [1]:
from pathlib import Path
import sys, subprocess

assert Path.cwd().name == "src", f"Abre este notebook desde SageMaker/src. Actual: {Path.cwd()}"
SCRIPT = Path("bts_sagemaker_endpoint.py").resolve()
assert SCRIPT.is_file(), "Sube bts_sagemaker_endpoint.py a SageMaker/src/"
import boto3
print("Python:", sys.executable)
print("Boto3:", boto3.__version__)
print("Script:", SCRIPT)

# Ejecutor independiente; cada comando inicia una sesión AWS propia.
def run_step(*args):
    result = subprocess.run([sys.executable, str(SCRIPT), *args], check=False)
    if result.returncode != 0:
        raise RuntimeError(f"El paso {args[0]} falló (código {result.returncode}); revisa el error mostrado")


Python: /home/ec2-user/anaconda3/envs/python3/bin/python
Boto3: 1.43.124
Script: /home/ec2-user/SageMaker/src/bts_sagemaker_endpoint.py


## 1. Validar el paquete que Optuna dejó en S3

Descarga temporalmente **solo** `model.tar.gz`, verifica SHA-256 si está registrado, comprueba los archivos de inferencia y recupera automáticamente la versión real desde `model_version.json`. **No rehace entrenamiento**. Si falta algo, vuelve a ejecutar la sección 11 del tuning (subida a S3).

In [2]:
run_step("preflight")

Cuenta IAM actual: arn:aws:sts::211125468543:assumed-role/LabRole/SageMaker

✔ S3: s3://bts-flights-analysis/models/flight-delay/lgbm-f64-seed42-h240-20261122T031926Z-trial-038/model.tar.gz
✔ Artefacto MiB: 4.48
✔ SHA-256: 1f2277547d650dd842b23378d1e29fbf62a2f00a207fe8863ecfd7299fe2bbfa
✔ Versión: lgbm-f64-seed42-h240-20261122T031926Z-trial-038
✔ Variables: 37 | Categóricas: 7
✔ Dependencia: lightgbm==4.7.0
✔ Umbral: 0.25
✔ Target: DepDel15
✔ Imagen administrada: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.9-0
✔ El endpoint espera características YA calculadas (horizonte 240 min).


## 2. Crear el modelo, la configuración y el endpoint

Usa la imagen **Scikit-learn 1.9-0** publicada por AWS, que cuenta con un entorno Python más moderno que la imagen obsoleta 1.2-1. Instala la versión de LightGBM declarada en el paquete `code/requirements.txt`. No crea un repositorio ECR propio.

Por defecto se solicita `ml.t3.medium` para reducir costos. Si el laboratorio no permite ese tipo **y la configuración aún no ha sido creada**, cambia el argumento a `--instance ml.m5.large` (sujeto también a permisos y cuotas). Si falla con `AccessDenied` en `iam:PassRole` o `CreateEndpoint`, es una restricción del laboratorio, no un error del modelo.

La creación es reanudable: si el recurso ya existe y corresponde a la misma versión, se reutiliza en lugar de duplicarse. El script **no reemplaza** automáticamente un endpoint que apunte a otra versión.

In [4]:
run_step("deploy", "--instance", "ml.m5.large")

Cuenta IAM actual: arn:aws:sts::211125468543:assumed-role/LabRole/SageMaker

✔ S3: s3://bts-flights-analysis/models/flight-delay/lgbm-f64-seed42-h240-20261122T031926Z-trial-038/model.tar.gz
✔ Artefacto MiB: 4.48
✔ SHA-256: 1f2277547d650dd842b23378d1e29fbf62a2f00a207fe8863ecfd7299fe2bbfa
✔ Versión: lgbm-f64-seed42-h240-20261122T031926Z-trial-038
✔ Variables: 37 | Categóricas: 7
✔ Dependencia: lightgbm==4.7.0
✔ Umbral: 0.25
✔ Target: DepDel15
✔ Imagen administrada: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.9-0
✔ El endpoint espera características YA calculadas (horizonte 240 min).
Versión del modelo: lgbm-f64-seed42-h240-20261122T031926Z-trial-038
Rol de ejecución: arn:aws:iam::211125468543:role/LabRole
Tipo de instancia: ml.m5.large
✔ Modelo ya existe, se reutiliza
Creando EndpointConfig: bts-delay-config-bb40ad10c5
✔ Configuración creada
Creando Endpoint: bts-flight-delay-api
✔ Solicitud de creación enviada. Consulta el estado con el comando status.

Endpoin

## 3. Consultar el estado (seguro tras reiniciar el laboratorio)

Una vez enviada la creación, espera hasta que el endpoint indique `InService`. Este comando es independiente; no necesitas volver a ejecutar `deploy` ni perder la sesión de notebook.

In [7]:
run_step("status")

Cuenta IAM actual: arn:aws:sts::211125468543:assumed-role/LabRole/SageMaker
23:13:51 bts-flight-delay-api → InService
✔ Puedes probar la API con: python bts_sagemaker_endpoint.py invoke


Opcional: consultar automáticamente el estado durante un máximo de 25 minutos. Si se interrumpe, vuelve a ejecutar `status`.

In [ ]:
# Descomenta si quieres esperar activamente:
# run_step("wait", "--max-minutes", "25")

## 4. Probar la API real y guardar evidencia en S3

El comando descarga `sample_request.json` correspondiente a la versión del modelo y llama a `sagemaker-runtime:InvokeEndpoint` con `ContentType=application/json`. Verifica la estructura de la respuesta y, si existe la predicción NPZ del tuning, compara la probabilidad contra la primera fila guardada (tolerancia `1e-5`).

Guarda el JSON de la prueba en:
- Local: `../outputs/outputs_modeling/deployment_sagemaker/endpoint_test_response.json`
- S3: `modeling/deployment/flight-delay/<model_version>/endpoint_test_response.json`

**No ejecutes hasta que el endpoint esté `InService`.** Esta celda es repetible después de reiniciar el laboratorio.

In [8]:
run_step("invoke")

Cuenta IAM actual: arn:aws:sts::211125468543:assumed-role/LabRole/SageMaker
23:13:57 bts-flight-delay-api → InService
✔ Puedes probar la API con: python bts_sagemaker_endpoint.py invoke

✔ Respuesta de la API:
{
  "model_version": "lgbm-f64-seed42-h240-20261122T031926Z-trial-038",
  "prediction_horizon_min": 240,
  "predictions": [
    {
      "probability_delay_15min": 0.1232693936551733,
      "alert": false
    }
  ],
  "decision_threshold": 0.25
}
Latencia cliente-medida (incluye red): 133.97 ms
✔ Coincide con el test guardado en tuning (|diferencia| ≤ 1e-5)
✔ Evidencia de consumo guardada en: s3://bts-flights-analysis/modeling/deployment/flight-delay/lgbm-f64-seed42-h240-20261122T031926Z-trial-038/endpoint_test_response.json


## 5. Cómo consumirlo desde otra aplicación

SageMaker expone una API HTTPS **autenticada con IAM**. No es un URL público que acepte solicitudes anónimas. Desde una Lambda con `LabRole`, una aplicación backend con credenciales AWS o un notebook, utiliza `boto3`:

```python
import boto3, json
client = boto3.client("sagemaker-runtime", region_name="us-east-1")
request = {"features": {"...": "..."}}  # Debe contener todas las variables del esquema
response = client.invoke_endpoint(
    EndpointName="bts-flight-delay-api",
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(request).encode("utf-8"),
)
print(json.loads(response["Body"].read()))
```

**Formato aceptado por tu `inference.py`:** `{"features": {...}}` o `{"instances": [{...}, {...}]}` (máximo 100). Las claves deben coincidir exactamente con `feature_schema.json`, salvo que se permita `null` en características numéricas; nunca incluyas la variable objetivo `DepDel15`.

**Formato de respuesta:** `model_version`, `prediction_horizon_min`, `decision_threshold` y `predictions` con probabilidad y `alert`.

Para un API REST expuesto a usuarios externos necesitaríamos además una Lambda intermediaria y API Gateway, con autenticación. No es obligatorio para demostrar consumo del endpoint en la rúbrica.


## 6. Diagnóstico de errores

- `AccessDenied` / `iam:PassRole`: revisar permisos y relación de confianza de `LabRole`; en AWS Academy algunas acciones pueden estar bloqueadas.
- `ResourceLimitExceeded` / instancia no permitida: la cuota del endpoint es distinta de la de la notebook `ml.t3.xlarge`.
- `UnexpectedStatusException` o `Failed`: SageMaker AI → Inference → Endpoints → `bts-flight-delay-api`, ver **Failure reason** y CloudWatch Logs (`/aws/sagemaker/Endpoints/bts-flight-delay-api`).
- `No module named lightgbm` / error pip: el contenedor no pudo instalar `code/requirements.txt`; revisar los logs y acceso saliente a PyPI. Puede requerirse empaquetado offline o imagen personalizada con LightGBM preinstalado.
- `Variables faltantes`: el JSON no está construido con el esquema exacto del entrenamiento; no inventar características.
- `ServiceUnavailable`: confirmar primero `InService` y que el contenedor arranque correctamente.

La imagen administrada se encuentra en el ECR público de AWS (cuenta registradora regional), **no es un ECR de tu cuenta**.


## 7. Eliminar endpoint tras la demostración — SOLO cuando lo autorices

**Este paso es destructivo.** Libera la capacidad de hosting y detiene el cobro del endpoint. Mantiene intacto el modelo versionado en S3 y los resultados del tuning. Se ofrece como comando para ejecutar *manualmente* cuando ya no requieras la API.

In [ ]:
# ATENCIÓN: habilitar solo cuando termines la demostración.
CONFIRMAR_ELIMINACION = False
if CONFIRMAR_ELIMINACION:
    run_step("cleanup", "--confirm")
else:
    print("No se eliminó ningún recurso. Cambia CONFIRMAR_ELIMINACION a True solo cuando termines.")